In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction import DictVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# PATHS
# ============================================================

INPUT_DIR = Path("/Users/user/XG/BEL/")
OUTPUT_DIR = Path("/Users/user/Downloads/xPass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# SETTINGS
# ============================================================

MIN_PLAYER_PASSES_FOR_IDENTITY = 50
PLAYER_ID_COL = "player_id_model"


# ============================================================
# HELPERS
# ============================================================

def qualifiers_to_dict(qualifiers):
    out = {}
    if not isinstance(qualifiers, list):
        return out

    for q in qualifiers:
        qid = q.get("qualifierId")
        if qid is None:
            continue
        out[qid] = q.get("value", True)

    return out


def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


# ============================================================
# PARSE ONE JSON FILE
# ============================================================

def parse_match_file(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    events = data.get("event", [])
    rows = []

    for ev in events:
        # Assumption: typeId == 1 corresponds to pass-like events
        if ev.get("typeId") != 1:
            continue

        qdict = qualifiers_to_dict(ev.get("qualifier", []))

        start_x = safe_float(ev.get("x"))
        start_y = safe_float(ev.get("y"))
        end_x = safe_float(qdict.get(140))
        end_y = safe_float(qdict.get(141))

        minute = safe_float(ev.get("timeMin"))
        second = safe_float(ev.get("timeSec"))

        if np.isnan(minute):
            minute = 0.0
        if np.isnan(second):
            second = 0.0

        rows.append(
            {
                "source_file": path.name,
                "match_id": path.stem,
                "team_id": ev.get("contestantId"),
                "player_id": ev.get("playerId"),
                "player_name": ev.get("playerName"),
                "period_id": ev.get("periodId"),
                "minute": minute,
                "second": second,
                "match_time": minute + second / 60.0,
                "outcome": ev.get("outcome"),
                "start_x": start_x,
                "start_y": start_y,
                "end_x": end_x,
                "end_y": end_y,
                "direction_str": qdict.get(56),
                "q212": safe_float(qdict.get(212)),
                "q213": safe_float(qdict.get(213)),
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.dropna(subset=["team_id", "player_id", "start_x", "start_y"])
    df = df[df["outcome"].isin([0, 1])].copy()

    return df


# ============================================================
# LOAD ALL JSON FILES IN FOLDER
# ============================================================

def load_all_matches(input_dir: Path) -> pd.DataFrame:
    files = sorted(input_dir.glob("*.json"))
    print(f"Found {len(files)} JSON files in {input_dir}")

    dfs = []
    failed = []

    for i, file in enumerate(files, start=1):
        try:
            df = parse_match_file(file)
            if not df.empty:
                dfs.append(df)
            if i % 25 == 0 or i == len(files):
                print(f"Processed {i}/{len(files)} files")
        except Exception as e:
            failed.append((str(file), str(e)))
            print(f"Failed to load {file}: {e}")

    if not dfs:
        raise ValueError(f"No usable pass data found in {input_dir}")

    combined = pd.concat(dfs, ignore_index=True)

    if failed:
        failed_df = pd.DataFrame(failed, columns=["file", "error"])
        failed_df.to_csv(OUTPUT_DIR / "failed_files.csv", index=False)
        print(f"Saved failed file log to {OUTPUT_DIR / 'failed_files.csv'}")

    return combined


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dx = df["end_x"] - df["start_x"]
    dy = df["end_y"] - df["start_y"]

    coord_length = np.sqrt(dx**2 + dy**2)
    df["pass_length"] = coord_length.where(~coord_length.isna(), df["q212"])
    df["pass_angle"] = np.arctan2(dy, dx)

    df["x_progress"] = dx
    df["y_progress"] = dy

    df["is_long"] = (df["pass_length"] >= 20).astype(float)
    df["is_forward"] = (df["x_progress"] > 0).astype(float)
    df["is_late"] = (df["match_time"] >= 75).astype(float)
    df["in_attacking_half"] = (df["start_x"] >= 50).astype(float)
    df["to_attacking_half"] = (df["end_x"] >= 50).fillna(0).astype(float)

    df["dir_back"] = (df["direction_str"] == "Back").astype(float)
    df["dir_left"] = (df["direction_str"] == "Left").astype(float)
    df["dir_right"] = (df["direction_str"] == "Right").astype(float)

    df["end_xy_missing"] = (df["end_x"].isna() | df["end_y"].isna()).astype(float)
    df["q212_missing"] = df["q212"].isna().astype(float)
    df["q213_missing"] = df["q213"].isna().astype(float)

    # Player exposure feature
    df["player_pass_count"] = df.groupby("player_id")["player_id"].transform("count").astype(float)

    # Group rare players to avoid overfitting
    player_counts = df["player_id"].value_counts()
    rare_players = player_counts[player_counts < MIN_PLAYER_PASSES_FOR_IDENTITY].index
    df[PLAYER_ID_COL] = df["player_id"].astype(str)
    df.loc[df["player_id"].isin(rare_players), PLAYER_ID_COL] = "OTHER"

    return df


# ============================================================
# PLAYER-SPECIFIC CONTEXT FEATURES
# ============================================================

def make_player_context_dicts(df: pd.DataFrame):
    rows = []

    for _, r in df.iterrows():
        pid = str(r[PLAYER_ID_COL])

        rows.append(
            {
                f"player_ctx__{pid}__forward": float(r["is_forward"]),
                f"player_ctx__{pid}__long": float(r["is_long"]),
                f"player_ctx__{pid}__late": float(r["is_late"]),
                f"player_ctx__{pid}__attacking_half": float(r["in_attacking_half"]),
            }
        )

    return rows


# ============================================================
# BUILD FEATURE MATRIX
# ============================================================

def build_feature_matrix(df: pd.DataFrame):
    y = df["outcome"].astype(int).to_numpy()

    numeric_cols = [
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "pass_length",
        "pass_angle",
        "x_progress",
        "y_progress",
        "match_time",
        "q212",
        "q213",
        "is_long",
        "is_forward",
        "is_late",
        "in_attacking_half",
        "to_attacking_half",
        "dir_back",
        "dir_left",
        "dir_right",
        "end_xy_missing",
        "q212_missing",
        "q213_missing",
        "player_pass_count",
    ]

    categorical_cols = [
        "team_id",
        PLAYER_ID_COL,
        "period_id",
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_cols,
            ),
        ],
        sparse_threshold=1.0,
    )

    X_base = preprocessor.fit_transform(df)

    interaction_dicts = make_player_context_dicts(df)
    dict_vectorizer = DictVectorizer(sparse=True)
    X_ctx = dict_vectorizer.fit_transform(interaction_dicts)

    X = sparse.hstack([X_base, X_ctx], format="csr")

    metadata = {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "preprocessor": preprocessor,
        "dict_vectorizer": dict_vectorizer,
    }

    return X, y, metadata


# ============================================================
# FIT MODEL
# ============================================================

def fit_model(df: pd.DataFrame):
    X, y, metadata = build_feature_matrix(df)

    model = LogisticRegression(
        penalty="l2",
        C=0.2,
        solver="liblinear",
        max_iter=2000,
        random_state=42,
    )

    model.fit(X, y)
    pred = model.predict_proba(X)[:, 1]

    metrics = {
        "n_passes": int(len(df)),
        "n_players_raw": int(df["player_id"].nunique()),
        "n_players_model": int(df[PLAYER_ID_COL].nunique()),
        "n_teams": int(df["team_id"].nunique()),
        "n_matches": int(df["match_id"].nunique()),
        "pass_success_rate": float(df["outcome"].mean()),
        "log_loss": float(log_loss(y, pred)),
        "brier_score": float(brier_score_loss(y, pred)),
        "roc_auc": float(roc_auc_score(y, pred)) if len(np.unique(y)) > 1 else np.nan,
    }

    print("\nModel metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.6f}")
        else:
            print(f"{k}: {v}")

    return model, pred, metadata, metrics


# ============================================================
# OUTPUT TABLES
# ============================================================

def summarize_passes(df: pd.DataFrame, pred: np.ndarray) -> pd.DataFrame:
    out = df.copy()
    out["xPass"] = pred
    out["over_under"] = out["outcome"] - out["xPass"]
    return out


def summarize_players(pass_df: pd.DataFrame) -> pd.DataFrame:
    player_summary = (
        pass_df.groupby(["player_id", "player_name", "team_id"], as_index=False)
        .agg(
            attempts=("outcome", "count"),
            completed=("outcome", "sum"),
            expected_completed=("xPass", "sum"),
            avg_xPass=("xPass", "mean"),
            player_pass_count=("player_pass_count", "max"),
            player_id_model=(PLAYER_ID_COL, "first"),
        )
    )

    player_summary["over_under_expected"] = (
        player_summary["completed"] - player_summary["expected_completed"]
    )
    player_summary["raw_completion_rate"] = (
        player_summary["completed"] / player_summary["attempts"]
    )

    return player_summary.sort_values(
        ["over_under_expected", "attempts"],
        ascending=[False, False]
    )


def extract_top_coefficients(model, metadata, top_n=100):
    preprocessor = metadata["preprocessor"]
    dict_vectorizer = metadata["dict_vectorizer"]

    base_names = preprocessor.get_feature_names_out()
    ctx_names = dict_vectorizer.feature_names_
    all_names = np.concatenate([base_names, ctx_names])

    coef = model.coef_[0]

    coef_df = pd.DataFrame(
        {
            "feature": all_names,
            "coefficient": coef,
            "abs_coefficient": np.abs(coef),
        }
    ).sort_values("abs_coefficient", ascending=False)

    return coef_df.head(top_n)


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    pass_summary: pd.DataFrame,
    player_summary: pd.DataFrame,
    metrics: dict,
    coef_summary: pd.DataFrame,
):
    pass_csv = OUTPUT_DIR / "pass_level.csv"
    player_csv = OUTPUT_DIR / "player_summary.csv"
    metrics_xlsx = OUTPUT_DIR / "model_metrics.xlsx"
    coef_csv = OUTPUT_DIR / "top_model_coefficients.csv"

    pass_summary.to_csv(pass_csv, index=False)
    player_summary.to_csv(player_csv, index=False)
    coef_summary.to_csv(coef_csv, index=False)

    metrics_df = pd.DataFrame([metrics])

    with pd.ExcelWriter(metrics_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(writer, sheet_name="metrics", index=False)

    print(f"\nSaved pass-level output to: {pass_csv}")
    print(f"Saved player summary to:   {player_csv}")
    print(f"Saved metrics Excel to:    {metrics_xlsx}")
    print(f"Saved coefficients to:     {coef_csv}")


# ============================================================
# MAIN
# ============================================================

def main():
    df = load_all_matches(INPUT_DIR)
    df = engineer_features(df)

    print(f"\nTotal passes used: {len(df)}")
    print(f"Matches used: {df['match_id'].nunique()}")
    print(f"Players used (raw): {df['player_id'].nunique()}")
    print(f"Players used (model IDs): {df[PLAYER_ID_COL].nunique()}")

    model, pred, metadata, metrics = fit_model(df)

    pass_summary = summarize_passes(df, pred)
    player_summary = summarize_players(pass_summary)
    coef_summary = extract_top_coefficients(model, metadata, top_n=100)

    save_outputs(pass_summary, player_summary, metrics, coef_summary)

    print("\nTop 10 players by over/under expected:")
    print(
        player_summary[
            [
                "player_name",
                "attempts",
                "completed",
                "expected_completed",
                "over_under_expected",
                "avg_xPass",
                "player_id_model",
            ]
        ].head(10)
    )

    print("\nTop 20 model coefficients:")
    print(coef_summary.head(20))


if __name__ == "__main__":
    main()

Found 224 JSON files in /Users/user/XG/BEL
Processed 25/224 files
Processed 50/224 files
Processed 75/224 files
Processed 100/224 files
Processed 125/224 files
Processed 150/224 files
Processed 175/224 files
Processed 200/224 files
Processed 224/224 files

Total passes used: 204699
Matches used: 224
Players used (raw): 446
Players used (model IDs): 374

Model metrics:
n_passes: 204699
n_players_raw: 446
n_players_model: 374
n_teams: 16
n_matches: 224
pass_success_rate: 0.778856
log_loss: 0.398005
brier_score: 0.128125
roc_auc: 0.833303

Saved pass-level output to: /Users/user/Downloads/xPass/pass_level.csv
Saved player summary to:   /Users/user/Downloads/xPass/player_summary.csv
Saved metrics Excel to:    /Users/user/Downloads/xPass/model_metrics.xlsx
Saved coefficients to:     /Users/user/Downloads/xPass/top_model_coefficients.csv

Top 10 players by over/under expected:
       player_name  attempts  completed  expected_completed  \
138     A. Diriken        37         33           26.

In [4]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction import DictVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# PATHS
# ============================================================

INPUT_DIR = Path("/Users/user/XG/BEL/")
OUTPUT_DIR = Path("/Users/user/Downloads/xPass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# SETTINGS
# ============================================================

MIN_PLAYER_PASSES_FOR_IDENTITY = 50
PLAYER_ID_COL = "player_id_model"


# ============================================================
# HELPERS
# ============================================================

def qualifiers_to_dict(qualifiers):
    out = {}
    if not isinstance(qualifiers, list):
        return out

    for q in qualifiers:
        qid = q.get("qualifierId")
        if qid is None:
            continue
        out[qid] = q.get("value", True)

    return out


def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


# ============================================================
# PARSE ONE JSON FILE
# ============================================================

def parse_match_file(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    events = data.get("event", [])
    rows = []

    for ev in events:
        # Assumption: typeId == 1 corresponds to pass-like events
        if ev.get("typeId") != 1:
            continue

        qdict = qualifiers_to_dict(ev.get("qualifier", []))

        start_x = safe_float(ev.get("x"))
        start_y = safe_float(ev.get("y"))
        end_x = safe_float(qdict.get(140))
        end_y = safe_float(qdict.get(141))

        minute = safe_float(ev.get("timeMin"))
        second = safe_float(ev.get("timeSec"))

        if np.isnan(minute):
            minute = 0.0
        if np.isnan(second):
            second = 0.0

        rows.append(
            {
                "source_file": path.name,
                "match_id": path.stem,
                "team_id": ev.get("contestantId"),
                "player_id": ev.get("playerId"),
                "player_name": ev.get("playerName"),
                "period_id": ev.get("periodId"),
                "minute": minute,
                "second": second,
                "match_time": minute + second / 60.0,
                "outcome": ev.get("outcome"),
                "start_x": start_x,
                "start_y": start_y,
                "end_x": end_x,
                "end_y": end_y,
                "direction_str": qdict.get(56),
                "q212": safe_float(qdict.get(212)),
                "q213": safe_float(qdict.get(213)),
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.dropna(subset=["team_id", "player_id", "start_x", "start_y"])
    df = df[df["outcome"].isin([0, 1])].copy()

    return df


# ============================================================
# LOAD ALL JSON FILES IN FOLDER
# ============================================================

def load_all_matches(input_dir: Path) -> pd.DataFrame:
    files = sorted(input_dir.glob("*.json"))
    print(f"Found {len(files)} JSON files in {input_dir}")

    dfs = []
    failed = []

    for i, file in enumerate(files, start=1):
        try:
            df = parse_match_file(file)
            if not df.empty:
                dfs.append(df)
            if i % 25 == 0 or i == len(files):
                print(f"Processed {i}/{len(files)} files")
        except Exception as e:
            failed.append((str(file), str(e)))
            print(f"Failed to load {file}: {e}")

    if not dfs:
        raise ValueError(f"No usable pass data found in {input_dir}")

    combined = pd.concat(dfs, ignore_index=True)

    if failed:
        failed_df = pd.DataFrame(failed, columns=["file", "error"])
        failed_df.to_csv(OUTPUT_DIR / "failed_files.csv", index=False)
        print(f"Saved failed file log to {OUTPUT_DIR / 'failed_files.csv'}")

    return combined


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dx = df["end_x"] - df["start_x"]
    dy = df["end_y"] - df["start_y"]

    coord_length = np.sqrt(dx**2 + dy**2)
    df["pass_length"] = coord_length.where(~coord_length.isna(), df["q212"])
    df["pass_angle"] = np.arctan2(dy, dx)

    df["x_progress"] = dx
    df["y_progress"] = dy

    df["is_long"] = (df["pass_length"] >= 20).astype(float)
    df["is_forward"] = (df["x_progress"] > 0).astype(float)
    df["is_late"] = (df["match_time"] >= 75).astype(float)
    df["in_attacking_half"] = (df["start_x"] >= 50).astype(float)
    df["to_attacking_half"] = (df["end_x"] >= 50).fillna(0).astype(float)

    df["dir_back"] = (df["direction_str"] == "Back").astype(float)
    df["dir_left"] = (df["direction_str"] == "Left").astype(float)
    df["dir_right"] = (df["direction_str"] == "Right").astype(float)

    df["end_xy_missing"] = (df["end_x"].isna() | df["end_y"].isna()).astype(float)
    df["q212_missing"] = df["q212"].isna().astype(float)
    df["q213_missing"] = df["q213"].isna().astype(float)

    # Player exposure feature
    df["player_pass_count"] = df.groupby("player_id")["player_id"].transform("count").astype(float)

    # Group rare players to avoid overfitting
    player_counts = df["player_id"].value_counts()
    rare_players = player_counts[player_counts < MIN_PLAYER_PASSES_FOR_IDENTITY].index
    df[PLAYER_ID_COL] = df["player_id"].astype(str)
    df.loc[df["player_id"].isin(rare_players), PLAYER_ID_COL] = "OTHER"

    return df


# ============================================================
# PLAYER-SPECIFIC CONTEXT FEATURES
# ============================================================

def make_player_context_dicts(df: pd.DataFrame):
    rows = []

    for _, r in df.iterrows():
        pid = str(r[PLAYER_ID_COL])

        rows.append(
            {
                f"player_ctx__{pid}__forward": float(r["is_forward"]),
                f"player_ctx__{pid}__long": float(r["is_long"]),
                f"player_ctx__{pid}__late": float(r["is_late"]),
                f"player_ctx__{pid}__attacking_half": float(r["in_attacking_half"]),
            }
        )

    return rows


# ============================================================
# BUILD FEATURE MATRIX
# ============================================================

def build_feature_matrix(df: pd.DataFrame):
    y = df["outcome"].astype(int).to_numpy()

    numeric_cols = [
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "pass_length",
        "pass_angle",
        "x_progress",
        "y_progress",
        "match_time",
        "q212",
        "q213",
        "is_long",
        "is_forward",
        "is_late",
        "in_attacking_half",
        "to_attacking_half",
        "dir_back",
        "dir_left",
        "dir_right",
        "end_xy_missing",
        "q212_missing",
        "q213_missing",
        "player_pass_count",
    ]

    categorical_cols = [
        "team_id",
        PLAYER_ID_COL,
        "period_id",
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_cols,
            ),
        ],
        sparse_threshold=1.0,
    )

    X_base = preprocessor.fit_transform(df)

    interaction_dicts = make_player_context_dicts(df)
    dict_vectorizer = DictVectorizer(sparse=True)
    X_ctx = dict_vectorizer.fit_transform(interaction_dicts)

    X = sparse.hstack([X_base, X_ctx], format="csr")

    metadata = {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "preprocessor": preprocessor,
        "dict_vectorizer": dict_vectorizer,
    }

    return X, y, metadata


# ============================================================
# FIT MODEL
# ============================================================

def fit_model(df: pd.DataFrame):
    X, y, metadata = build_feature_matrix(df)

    model = LogisticRegression(
        penalty="l2",
        C=0.2,
        solver="liblinear",
        max_iter=2000,
        random_state=42,
    )

    model.fit(X, y)
    pred = model.predict_proba(X)[:, 1]

    metrics = {
        "n_passes": int(len(df)),
        "n_players_raw": int(df["player_id"].nunique()),
        "n_players_model": int(df[PLAYER_ID_COL].nunique()),
        "n_teams": int(df["team_id"].nunique()),
        "n_matches": int(df["match_id"].nunique()),
        "pass_success_rate": float(df["outcome"].mean()),
        "log_loss": float(log_loss(y, pred)),
        "brier_score": float(brier_score_loss(y, pred)),
        "roc_auc": float(roc_auc_score(y, pred)) if len(np.unique(y)) > 1 else np.nan,
    }

    print("\nModel metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.6f}")
        else:
            print(f"{k}: {v}")

    return model, pred, metadata, metrics


# ============================================================
# OUTPUT TABLES
# ============================================================

def summarize_passes(df: pd.DataFrame, pred: np.ndarray) -> pd.DataFrame:
    out = df.copy()
    out["xPass"] = pred
    out["over_under"] = out["outcome"] - out["xPass"]
    return out


def summarize_players(pass_df: pd.DataFrame) -> pd.DataFrame:
    player_summary = (
        pass_df.groupby(["player_id", "player_name", "team_id"], as_index=False)
        .agg(
            attempts=("outcome", "count"),
            completed=("outcome", "sum"),
            expected_completed=("xPass", "sum"),
            avg_xPass=("xPass", "mean"),
            player_pass_count=("player_pass_count", "max"),
            player_id_model=(PLAYER_ID_COL, "first"),
        )
    )

    player_summary["over_under_expected"] = (
        player_summary["completed"] - player_summary["expected_completed"]
    )
    player_summary["raw_completion_rate"] = (
        player_summary["completed"] / player_summary["attempts"]
    )

    return player_summary.sort_values(
        ["over_under_expected", "attempts"],
        ascending=[False, False]
    )


def make_player_export(player_summary: pd.DataFrame) -> pd.DataFrame:
    export_df = player_summary.copy()

    export_df["Average pass %"] = export_df["raw_completion_rate"] * 100.0
    export_df["xPass per pass"] = export_df["avg_xPass"]
    export_df["Average pass difficulty"] = 1.0 - export_df["avg_xPass"]

    export_df = export_df.rename(
        columns={
            "player_name": "playerName",
            "team_id": "contestantId",
            "attempts": "Passes",
            "completed": "Accurate passes",
            "expected_completed": "xPass",
        }
    )

    export_df = export_df[
        [
            "playerName",
            "contestantId",
            "Passes",
            "Accurate passes",
            "Average pass %",
            "xPass",
            "xPass per pass",
            "Average pass difficulty",
        ]
    ].sort_values(["xPass", "Passes"], ascending=[False, False])

    return export_df


def extract_top_coefficients(model, metadata, top_n=100):
    preprocessor = metadata["preprocessor"]
    dict_vectorizer = metadata["dict_vectorizer"]

    base_names = preprocessor.get_feature_names_out()
    ctx_names = dict_vectorizer.feature_names_
    all_names = np.concatenate([base_names, ctx_names])

    coef = model.coef_[0]

    coef_df = pd.DataFrame(
        {
            "feature": all_names,
            "coefficient": coef,
            "abs_coefficient": np.abs(coef),
        }
    ).sort_values("abs_coefficient", ascending=False)

    return coef_df.head(top_n)


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    pass_summary: pd.DataFrame,
    player_summary: pd.DataFrame,
    player_export: pd.DataFrame,
    metrics: dict,
    coef_summary: pd.DataFrame,
):
    pass_csv = OUTPUT_DIR / "pass_level.csv"
    player_csv = OUTPUT_DIR / "player_summary.csv"
    metrics_xlsx = OUTPUT_DIR / "model_metrics.xlsx"
    coef_csv = OUTPUT_DIR / "top_model_coefficients.csv"
    player_export_xlsx = OUTPUT_DIR / "player_pass_export.xlsx"

    pass_summary.to_csv(pass_csv, index=False)
    player_summary.to_csv(player_csv, index=False)
    coef_summary.to_csv(coef_csv, index=False)

    metrics_df = pd.DataFrame([metrics])

    with pd.ExcelWriter(metrics_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(writer, sheet_name="metrics", index=False)
        player_summary.to_excel(writer, sheet_name="player_summary", index=False)
        coef_summary.to_excel(writer, sheet_name="coefficients", index=False)

    with pd.ExcelWriter(player_export_xlsx, engine="openpyxl") as writer:
        player_export.to_excel(writer, sheet_name="player_export", index=False)

    print(f"\nSaved pass-level output to: {pass_csv}")
    print(f"Saved player summary to:   {player_csv}")
    print(f"Saved metrics Excel to:    {metrics_xlsx}")
    print(f"Saved coefficients to:     {coef_csv}")
    print(f"Saved player export to:    {player_export_xlsx}")


# ============================================================
# MAIN
# ============================================================

def main():
    df = load_all_matches(INPUT_DIR)
    df = engineer_features(df)

    print(f"\nTotal passes used: {len(df)}")
    print(f"Matches used: {df['match_id'].nunique()}")
    print(f"Players used (raw): {df['player_id'].nunique()}")
    print(f"Players used (model IDs): {df[PLAYER_ID_COL].nunique()}")

    model, pred, metadata, metrics = fit_model(df)

    pass_summary = summarize_passes(df, pred)
    player_summary = summarize_players(pass_summary)
    player_export = make_player_export(player_summary)
    coef_summary = extract_top_coefficients(model, metadata, top_n=100)

    save_outputs(pass_summary, player_summary, player_export, metrics, coef_summary)

    print("\nTop 10 players by over/under expected:")
    print(
        player_summary[
            [
                "player_name",
                "attempts",
                "completed",
                "expected_completed",
                "over_under_expected",
                "avg_xPass",
                "player_id_model",
            ]
        ].head(10)
    )

    print("\nTop 20 model coefficients:")
    print(coef_summary.head(20))

    print("\nTop 10 rows of player export:")
    print(player_export.head(10))


if __name__ == "__main__":
    main()

Found 224 JSON files in /Users/user/XG/BEL
Processed 25/224 files
Processed 50/224 files
Processed 75/224 files
Processed 100/224 files
Processed 125/224 files
Processed 150/224 files
Processed 175/224 files
Processed 200/224 files
Processed 224/224 files

Total passes used: 204699
Matches used: 224
Players used (raw): 446
Players used (model IDs): 374

Model metrics:
n_passes: 204699
n_players_raw: 446
n_players_model: 374
n_teams: 16
n_matches: 224
pass_success_rate: 0.778856
log_loss: 0.398005
brier_score: 0.128125
roc_auc: 0.833303

Saved pass-level output to: /Users/user/Downloads/xPass/pass_level.csv
Saved player summary to:   /Users/user/Downloads/xPass/player_summary.csv
Saved metrics Excel to:    /Users/user/Downloads/xPass/model_metrics.xlsx
Saved coefficients to:     /Users/user/Downloads/xPass/top_model_coefficients.csv
Saved player export to:    /Users/user/Downloads/xPass/player_pass_export.xlsx

Top 10 players by over/under expected:
       player_name  attempts  complet